# Train & Plot: Your First IMPALA Experiment

Smoke-test the training pipeline and visualize training results.

**What you'll do**:
1. Smoke test (30 seconds) — verify everything works
2. Load and plot training logs from any experiment
3. Visualize learning curves and loss components

**Requirements**: `pip install torch matplotlib`  
**Time**: ~2 minutes

In [ ]:
%matplotlib inline

In [ ]:
import sys, os

import pathlib
PROJECT_ROOT = str(pathlib.Path.cwd().parent) if pathlib.Path('../train_impala.py').exists() else str(pathlib.Path.cwd())
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'notebooks'))
os.environ.setdefault("MUJOCO_GL", "egl" if sys.platform == "linux" else "glfw")

import subprocess
import numpy as np
import matplotlib.pyplot as plt

from utils import load_training_logs, logs_to_arrays, plot_learning_curve, plot_loss_components, smooth

## 1. Smoke Test

First, verify that training runs without errors. This takes ~30 seconds and only runs 1000 steps with 2 actors.

In [ ]:
result = subprocess.run(
    [sys.executable, "train_impala.py",
     "--num_actors", "2",
     "--total_steps", "1000",
     "--batch_size", "2",
     "--savedir", "/tmp/memorymaze_smoke",
     "--xpid", "smoke_test",
     "--disable_checkpoint"],
    cwd=PROJECT_ROOT,
    capture_output=True, text=True, timeout=120
)

if result.returncode == 0:
    print("Smoke test PASSED")
    # Show last few lines
    for line in result.stderr.strip().split("\n")[-5:]:
        print(f"  {line}")
else:
    print("Smoke test FAILED")
    print(result.stderr[-1000:])

## 2. Load Training Logs

Load and plot logs from any completed experiment. Logs are auto-discovered from the log directory.

In [ ]:
LOGDIR = os.path.expanduser('~/logs/torchbeast')

# Auto-discover available experiments
print(f"Log directory: {LOGDIR}\n")
available = []
if os.path.exists(LOGDIR):
    for d in sorted(os.listdir(LOGDIR)):
        logs_path = os.path.join(LOGDIR, d, 'logs.csv')
        if os.path.exists(logs_path):
            size_kb = os.path.getsize(logs_path) / 1024
            available.append(d)
            print(f"  {d:40s}  (logs.csv: {size_kb:.1f} KB)")

if not available:
    print("  No experiments found yet.")
    print("  Run training first, then come back here to plot.")
else:
    print(f"\n{len(available)} experiment(s) found.")

# Set XPID to the experiment you want to plot (defaults to most recent)
XPID = available[-1] if available else "impala_9x9_demo"
print(f"\nSelected XPID: {XPID}")

## 3. Plot Learning Curve

Load the training logs and plot episode returns over time.

In [ ]:
rows = load_training_logs(XPID, logdir=LOGDIR)
steps, returns = logs_to_arrays(rows, step_key="step", value_key="mean_episode_return")

print(f"Loaded {len(rows)} log entries")
print(f"Step range: {steps[0]:,.0f} - {steps[-1]:,.0f}")
print(f"Return range: {returns.min():.1f} - {returns.max():.1f}")

fig, ax = plt.subplots(figsize=(10, 5))
plot_learning_curve(steps, returns, label="IMPALA 9x9", smooth_window=20, ax=ax,
                    title="IMPALA Learning Curve — Memory Maze 9x9")
plt.tight_layout()
plt.savefig("learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: learning_curve.png")

## 4. Loss Components

Visualize the individual loss terms: policy gradient loss, baseline (value) loss, and entropy.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Total loss
steps, total_loss = logs_to_arrays(rows, value_key="total_loss")
axes[0, 0].plot(steps, smooth(total_loss, 20), linewidth=1.5)
axes[0, 0].set_title("Total Loss")
axes[0, 0].set_xlabel("Steps")
axes[0, 0].grid(True, alpha=0.3)

# Policy gradient loss
_, pg_loss = logs_to_arrays(rows, value_key="pg_loss")
axes[0, 1].plot(steps[:len(pg_loss)], smooth(pg_loss, 20), linewidth=1.5, color="tab:orange")
axes[0, 1].set_title("Policy Gradient Loss")
axes[0, 1].set_xlabel("Steps")
axes[0, 1].grid(True, alpha=0.3)

# Baseline loss
_, bl_loss = logs_to_arrays(rows, value_key="baseline_loss")
axes[1, 0].plot(steps[:len(bl_loss)], smooth(bl_loss, 20), linewidth=1.5, color="tab:green")
axes[1, 0].set_title("Baseline (Value) Loss")
axes[1, 0].set_xlabel("Steps")
axes[1, 0].grid(True, alpha=0.3)

# Entropy loss
_, ent_loss = logs_to_arrays(rows, value_key="entropy_loss")
axes[1, 1].plot(steps[:len(ent_loss)], smooth(ent_loss, 20), linewidth=1.5, color="tab:red")
axes[1, 1].set_title("Entropy Loss")
axes[1, 1].set_xlabel("Steps")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("IMPALA Loss Components", fontsize=14)
plt.tight_layout()
plt.savefig("loss_components.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Training Diagnostics

Additional metrics: gradient norms, LSTM hidden state norms, and episodes per batch.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Grad norm
_, grad_norm = logs_to_arrays(rows, value_key="grad_norm")
axes[0].plot(steps[:len(grad_norm)], smooth(grad_norm, 20), linewidth=1.5)
axes[0].set_title("Gradient Norm")
axes[0].set_xlabel("Steps")
axes[0].grid(True, alpha=0.3)

# LSTM hidden norm
_, lstm_norm = logs_to_arrays(rows, value_key="lstm_hidden_norm")
axes[1].plot(steps[:len(lstm_norm)], smooth(lstm_norm, 20), linewidth=1.5, color="tab:purple")
axes[1].set_title("LSTM Hidden State Norm")
axes[1].set_xlabel("Steps")
axes[1].grid(True, alpha=0.3)

# Episodes per batch
_, epb = logs_to_arrays(rows, value_key="episodes_per_batch")
axes[2].plot(steps[:len(epb)], smooth(epb, 20), linewidth=1.5, color="tab:brown")
axes[2].set_title("Episodes per Batch")
axes[2].set_xlabel("Steps")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Compare Multiple Experiments

If you have multiple training runs (e.g., MuJoCo vs Genesis), overlay their learning curves.
Set the XPIDs below, or leave empty to auto-discover all available experiments.

In [ ]:
XPIDS_TO_COMPARE = []  # e.g. ['impala_mujoco', 'impala_genesis'] — leave empty to auto-discover

if not XPIDS_TO_COMPARE:
    XPIDS_TO_COMPARE = available  # from cell 6

if len(XPIDS_TO_COMPARE) < 2:
    print(f"Need at least 2 experiments to compare. Found: {XPIDS_TO_COMPARE}")
    print("Run more training experiments, then come back.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.tab10.colors

    for i, xpid in enumerate(XPIDS_TO_COMPARE):
        try:
            rows_c = load_training_logs(xpid, logdir=LOGDIR)
            steps_c, returns_c = logs_to_arrays(rows_c, value_key="mean_episode_return")
            plot_learning_curve(steps_c, returns_c, label=xpid, color=colors[i % len(colors)],
                                smooth_window=20, ax=ax)
        except Exception as e:
            print(f"  {xpid}: {e}")

    ax.set_title("Learning Curves Comparison")
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig("comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## Next Steps

- **Evaluate checkpoint**: See `03_evaluate_and_record.ipynb`
- **Engine comparison**: See `04_engine_comparison.ipynb`
- **Full paper replication**: Use 128 actors and 100M steps (needs GPU, ~24 hours)